# GazeRefine — Interactive Walkthrough

This notebook walks through the **full GazeRefine pipeline** step by step:

1. Build a gaze heatmap from a scanpath
2. Construct gaze-anchored foreground/background prototypes in frozen DINOv3 feature space
3. Run the recurrent refinement loop (contrastive cleaning + kNN affinity propagation)
4. Visualize the final mask

It ships with a **synthetic toy example** (a blob on a noisy background, with a few clicked
fixation points) so you can see every intermediate step without needing any dataset.
Swap in a real image + fixation CSV from `examples/` (see `examples/README.md`) to run it
on actual data — the API is identical.

> Section 4 needs `torch` + `timm` (and a DINOv3 checkpoint download, so internet access)
> to call the real backbone. Sections 1–3 only need `numpy`/`matplotlib` and run anywhere,
> using a tiny **mock feature extractor** so you can still see the prototype/refinement math.

In [ ]:
import sys, os
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
%matplotlib inline

## 1. A synthetic example

We draw a soft circular "lesion" on a noisy background and pick a few fixation points near
its center — mimicking a clinician glancing at the structure of interest a handful of times,
with the first, longest fixation landing closest to the center.

In [ ]:
IMG_SIZE = 224  # small, just for a fast visual demo (the real pipeline defaults to 1184 / DINOv3 patch grid)

def make_toy_image(size=IMG_SIZE, cx=0.55, cy=0.45, r=0.12):
    yy, xx = np.mgrid[0:size, 0:size] / size
    dist = np.sqrt((xx - cx) ** 2 + (yy - cy) ** 2)
    blob = np.clip(1 - dist / r, 0, 1) ** 2
    background = 0.25 + 0.05 * np.random.randn(size, size)
    img = np.clip(background + 0.6 * blob, 0, 1)
    gt_mask = (dist < r).astype(np.float32)
    return img, gt_mask

toy_img, toy_mask = make_toy_image()

# a few fixations clustered near the blob, decaying duration, with a touch of natural jitter

fixations = [
    (0.55, 0.45, 1.0),
    (0.57, 0.47, 0.7),
    (0.53, 0.44, 0.5),
    (0.50, 0.50, 0.3),
]

fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(toy_img, cmap="gray"); ax[0].set_title("Synthetic image")
for x, y, d in fixations:
    ax[0].scatter(x * IMG_SIZE, y * IMG_SIZE, s=200 * d, c="red", alpha=0.6, edgecolors="white")
ax[1].imshow(toy_mask, cmap="gray"); ax[1].set_title("Ground-truth mask (synthetic)")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

## 2. Scanpath → gaze heatmap

`gazerefine.gaze.generate_gaze_heatmap` places a duration-weighted Gaussian at each fixation
and min-max normalizes the result. This becomes the soft foreground prior $W_{fg}^{(0)}$;
$1 - H$ becomes the background prior $W_{bg}^{(0)}$.

In [ ]:
import torch
from gazerefine.gaze import generate_gaze_heatmap

H_PATCH = 28  # toy patch grid (real pipeline uses 74x74, see gazerefine/constants.py)
fixation_t = torch.tensor([fixations], dtype=torch.float32)  # (1, L, 3)

gaze_heatmap = generate_gaze_heatmap(fixation_t, h_patch=H_PATCH, sigma=2.0)[0].numpy()

plt.figure(figsize=(4, 4))
plt.imshow(gaze_heatmap, cmap="jet")
plt.title("Duration-weighted gaze heatmap (patch grid)")
plt.axis("off"); plt.colorbar(fraction=0.046); plt.show()

## 3. Prototype construction & refinement — mechanics demo

To see the **math** of gaze-anchored prototypes, contrastive cleaning, and kNN affinity
propagation without downloading a backbone, we substitute a tiny mock "feature extractor":
patches near the blob get embeddings clustered around one random direction, background
patches cluster around another, with noise — a cartoon stand-in for "DINOv3 already groups
semantically similar patches together", which is the real property GazeRefine exploits.

**This section is for intuition only.** Section 4 below runs the real frozen DINOv3 backbone.

In [ ]:
from gazerefine.model import knn_affinity_refinement


D = 64  # toy embedding dim

rng = np.random.default_rng(0)

fg_direction = rng.normal(size=D); fg_direction /= np.linalg.norm(fg_direction)

bg_direction = rng.normal(size=D); bg_direction /= np.linalg.norm(bg_direction)



# resize the toy GT mask down to the patch grid to decide which mock cluster each patch belongs to

import torch.nn.functional as Fnn

mask_small = Fnn.interpolate(torch.tensor(toy_mask)[None, None], size=(H_PATCH, H_PATCH), mode="nearest")[0, 0].numpy()

patch_is_fg = mask_small.flatten() > 0.5



N = H_PATCH * H_PATCH

Pv = np.stack([

    (fg_direction if patch_is_fg[i] else bg_direction) + 0.3 * rng.normal(size=D)

    for i in range(N)

])

Pv = torch.tensor(Pv, dtype=torch.float32)[None]  # (1, N, D)


In [ ]:
H_flat = torch.tensor(gaze_heatmap.flatten(), dtype=torch.float32)[None]  # (1, N)

eps = 1e-12

W_fg = H_flat / (H_flat.sum(-1, keepdim=True) + eps)

W_bg = (1 - H_flat) / (1 - H_flat).sum(-1, keepdim=True)



Pv_norm = Fnn.normalize(Pv, p=2, dim=-1)

F_proto = Fnn.normalize((W_fg.unsqueeze(-1) * Pv).sum(1), p=2, dim=-1)

B_proto = Fnn.normalize((W_bg.unsqueeze(-1) * Pv).sum(1), p=2, dim=-1)



sim_fg = torch.bmm(Pv_norm, F_proto.unsqueeze(-1)).squeeze(-1)

sim_bg = torch.bmm(Pv_norm, B_proto.unsqueeze(-1)).squeeze(-1)

S_clean = torch.clamp(sim_fg - sim_bg, min=0.0)              # contrastive cleaning

S_knn = knn_affinity_refinement(Pv, S_clean, k=8, temperature=0.1)  # kNN affinity propagation



def to_grid(s):

    g = s[0].reshape(H_PATCH, H_PATCH).detach().numpy()

    return (g - g.min()) / (g.max() - g.min() + 1e-8)



fig, ax = plt.subplots(1, 4, figsize=(16, 4))

ax[0].imshow(gaze_heatmap, cmap="jet"); ax[0].set_title("Gaze prior H")

ax[1].imshow(to_grid(sim_fg - sim_bg), cmap="viridis"); ax[1].set_title("sim_fg − sim_bg (pre-clean)")

ax[2].imshow(to_grid(S_clean), cmap="viridis"); ax[2].set_title("After contrastive cleaning")

ax[3].imshow(to_grid(S_knn), cmap="viridis"); ax[3].set_title("After kNN propagation")

for a in ax: a.axis("off")

plt.tight_layout(); plt.show()


Notice how the gaze prior alone is a soft, blurry blob centered on the clicked points,

but after contrastive cleaning + kNN propagation the response sharpens to the full extent

of the (mock-)semantically coherent foreground region — this is the localization gain

described in the paper, beyond the initially fixated patches.

## 4. The real pipeline — frozen DINOv3 + `GazeRefine`

This cell calls the actual model end to end. It needs `torch`, `timm`, and internet access
the first time (to download the DINOv3 checkpoint). If those aren't available in your current
environment, this cell will print instructions instead of failing silently — run it in a
GPU-enabled environment with the repo's `requirements.txt` installed.

In [ ]:
try:

    from gazerefine import GazeRefine

    from gazerefine.constants import IMG_SIZE

    import torchvision.transforms as T



    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



    model = GazeRefine(

        dino_name="vit_large_patch16_dinov3.lvd1689m",

        sigma=2.0, contrast_method="difference", max_iters=5,

        gaze_anchor_weight=0.5, knn_k=20,

    ).to(device).eval()



    img_3ch = np.stack([toy_img] * 3, axis=-1)

    tf = T.Compose([T.ToPILImage(), T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor(),

                     T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

    img_t = tf((img_3ch * 255).astype(np.uint8)).unsqueeze(0).to(device)



    fix_t = torch.tensor([fixations], dtype=torch.float32).to(device)



    with torch.no_grad():

        out = model(img_t, fix_t)



    pred = out["preds"][0, 0].cpu().numpy()

    fig, ax = plt.subplots(1, 3, figsize=(12, 4))

    ax[0].imshow(toy_img, cmap="gray"); ax[0].set_title("Input")

    ax[1].imshow(out["gaze_heatmap"][0].cpu().numpy(), cmap="jet"); ax[1].set_title("Gaze prior")

    ax[2].imshow(pred > 0.5, cmap="gray"); ax[2].set_title("GazeRefine mask")

    for a in ax: a.axis("off")

    plt.tight_layout(); plt.show()



except ModuleNotFoundError as e:

    print(f"[skipped] missing dependency: {e}")

    print("Install requirements.txt (torch, timm, ...) and re-run this cell to see real DINOv3 output.")

except Exception as e:

    print(f"[skipped] could not run the real backbone in this environment: {e}")

    print("This is most likely a missing internet connection for the first-time checkpoint download.")


## 5. Try it on a real image + fixation CSV

Once you've added real assets under `examples/` (see `examples/README.md`), point
`scripts.predict_single.predict` at them directly:

In [ ]:
# from scripts.predict_single import predict

#

# out = predict(

#     image="../examples/images/kvasir_sample_01.jpg",

#     fixation_csv="../examples/fixations/kvasir_sample_01_fixations.csv",

#     image_key="kvasir_sample_01.jpg",

#     sigma=2.0, contrast_method="difference", max_iters=5,

#     gaze_anchor_weight=0.5, knn_k=20,

# )

# out["mask_overlay"]
